In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   80K   20G   1% /kaggle/working


In [3]:
!du -sh /kaggle/working/* | sort -rh

du: cannot access '/kaggle/working/*': No such file or directory


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Forces the script to only see one GPU

In [5]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [6]:
from unsloth import FastLanguageModel
# Prevention for memory fragmentation



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
max_seq_length = 2048 # Adjust based on legal document lengths
dtype = None # Auto-detection
load_in_4bit = True # Essential for T4 GPUs

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters for DAPT
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Higher rank is often better for DAPT to absorb new knowledge
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [8]:
from datasets import load_dataset

# Load your uploaded Parquet file
# /kaggle/input/datasets/rehanfargose/sc-dataset-1990-2025
dataset = load_dataset("parquet", data_files={"train": "/kaggle/input/datasets/rehanfargose/sc-dataset-1990-2025/SC_Parquet_Dataset_1990-2025.parquet"}, split="train")

def formatting_prompts_func(examples):
    # 'full_text' is the column name in parquet
    return { "text" : [t + tokenizer.eos_token for t in examples["full_text"]] }

dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/26518 [00:00<?, ? examples/s]

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments



In [10]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, 
    dataset_batch_size = 100,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Keep at 1
        gradient_accumulation_steps = 8, # Increase if you reduce batch size
        warmup_steps = 50,
        # Re-calculate steps based on new batching logic
        # max_steps = (len(dataset) // (1 * 8)), 
        max_steps = 100, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        gradient_checkpointing = True,
        weight_decay = 0.01,
        save_strategy = "no",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/26518 [00:00<?, ? examples/s]

In [11]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,518 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
1,2.823714
2,2.862135
3,2.711598
4,2.724232
5,2.632213
6,2.675095
7,2.638793
8,2.692291
9,2.586769
10,2.565370


TrainOutput(global_step=100, training_loss=2.140511705875397, metrics={'train_runtime': 3973.7886, 'train_samples_per_second': 0.201, 'train_steps_per_second': 0.025, 'total_flos': 3.719192672636928e+16, 'train_loss': 2.140511705875397, 'epoch': 0.030168187646127158})

In [12]:
# Save the model after DAPT
# model.save_pretrained_merged("Deepseek_R1_8B_DAPT", tokenizer, save_method = "merged_16bit")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:51<00:00, 12.77s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:17<00:00, 34.32s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Deepseek_R1_8B_DAPT`


In [14]:
# This saves ONLY the trained adapter layers (very small and disk-safe)
model.save_pretrained("Deepseek_R1_8B_DAPT_LoRA")
tokenizer.save_pretrained("Deepseek_R1_8B_DAPT_LoRA")

('Deepseek_R1_8B_DAPT_LoRA/tokenizer_config.json',
 'Deepseek_R1_8B_DAPT_LoRA/chat_template.jinja',
 'Deepseek_R1_8B_DAPT_LoRA/tokenizer.json')

In [15]:
import shutil
from IPython.display import FileLink

# Zip the small adapter folder
shutil.make_archive("Deepseek_R1_8B_DAPT_LoRA", 'zip', "Deepseek_R1_8B_DAPT_LoRA")

# Generate the download link
FileLink(r'Deepseek_R1_8B_DAPT_LoRA.zip')

/kaggle/working/Deepseek_R1_8B_DAPT_LoRA.zip